# Yu (Astronet-Triage original) vs. Astronet-Triage v2 — test-set comparison

Both models are scored on the same v2 test TFRecords. The v2 model emits per-class probabilities (`disp_e/n/j/s/b`); the Yu model is binary with `disp_E > 0` as the positive class. Here we compare them on that shared binary axis.

**Yu predictions are produced out-of-notebook.** Before running this notebook, score the Yu ensemble in WSL once:
```bash
cd /mnt/c/Users/danie/Documents/personal/test/astroyul/Astronet-Triage
PYTHONPATH=. python \
    /mnt/c/Users/danie/Documents/personal/test/Astronet-Triage/scripts/predict_yu_ensemble.py \
    --model_root runs/fa1t_yu \
    --tfrecord_glob '/mnt/c/Users/danie/Documents/personal/test/Astronet-Triage/data/tfrecords/test/*' \
    --output_file /mnt/c/Users/danie/Documents/personal/test/Astronet-Triage/data/yu_ensemble_test.csv
```
That writes a single CSV the notebook reads below.

In [ ]:
import os

V2_CHKPT_ROOT = 'checkpoints/fa1t_38_run_2'
V2_DATA_FILES = 'data/tfrecords/test/*'
V2_TCES_FILE = 'data/tces-v14-test.csv'
YU_PREDS_FILE = 'data/yu_ensemble_test.csv'

NRUNS = 10

## 1. v2 ensemble predictions (inline)
Same pattern as `Predict-fa1t-38-test.ipynb`.

In [ ]:
def load_v2_ensemble(root, nruns):
    paths = []
    for i in range(nruns):
        parent = os.path.join(root, str(i + 1))
        if not os.path.exists(parent):
            break
        subs = os.listdir(parent)
        if not subs:
            break
        d, = subs
        paths.append(os.path.join(parent, d))
    return paths

v2_paths = load_v2_ensemble(V2_CHKPT_ROOT, NRUNS)
v2_paths

In [ ]:
from astronet import predict as v2_predict


def run_v2(path):
    v2_predict.FLAGS = v2_predict.parser.parse_args([
        '--model_dir', path,
        '--data_files', V2_DATA_FILES,
        '--output_file', '',
    ])
    return v2_predict.predict()


v2_ensemble_preds = []
v2_config = None
for i, path in enumerate(v2_paths):
    print(f'v2 model {i + 1}')
    preds, v2_config = run_v2(path)
    v2_ensemble_preds.append(preds.set_index('astro_id'))
    print()

## 2. Yu ensemble predictions (from pre-computed file)

In [ ]:
import pandas as pd

yu_df = pd.read_csv(YU_PREDS_FILE).set_index('astro_id')
yu_pred_cols = [c for c in yu_df.columns if c.startswith('yu_pred_')]
print(f'Yu CSV: {len(yu_df)} rows, {len(yu_pred_cols)} ensemble members')
yu_df.head(3)

## 3. Ground truth and shared id set

In [ ]:
import numpy as np

labels = ['disp_e', 'disp_n', 'disp_j', 'disp_s', 'disp_b']

tce_table = pd.read_csv(V2_TCES_FILE, header=0, low_memory=False)
tce_table['astro_id'] = tce_table['Astro ID']
tce_table = tce_table.set_index('astro_id')
for l in labels:
    tce_table[l] = tce_table[l[:-1] + l[-1].upper()]
tce_labels = tce_table[labels + ['TIC ID']]

ids = sorted(set(yu_df.index) & set(v2_ensemble_preds[0].index))
index = {v: i for i, v in enumerate(ids)}

lbl_e = np.zeros(len(ids), dtype=bool)
for ex_id, row in tce_labels.iterrows():
    if ex_id in index:
        lbl_e[index[ex_id]] = (row['disp_e'] > 0)
print(f'{len(ids)} shared TCEs, {int(lbl_e.sum())} positives')

In [ ]:
col_e = labels.index('disp_e')

# v2: average disp_e across 10 ensemble DataFrames
v2_mat = np.zeros([len(v2_ensemble_preds), len(ids)])
for i, preds in enumerate(v2_ensemble_preds):
    for ex_id, row in preds.iterrows():
        if ex_id in index:
            v2_mat[i][index[ex_id]] = row.iloc[col_e]

# Yu: already wide (yu_pred_1..N), aligned by astro_id
yu_mat = np.zeros([len(yu_pred_cols), len(ids)])
for i, col in enumerate(yu_pred_cols):
    s = yu_df[col]
    for ex_id in ids:
        yu_mat[i][index[ex_id]] = s.loc[ex_id]

v2_mean = v2_mat.mean(axis=0)
yu_mean = yu_mat.mean(axis=0)
v2_mat.shape, yu_mat.shape

## 4. Precision-recall curves

In [ ]:
from matplotlib import pyplot as plt
from sklearn import metrics


def pr_sweep(scores, labels_bool):
    num_cond_pos = int(labels_bool.sum())
    ps, rs, ths = [], [], []
    th = float(scores.max())
    while th >= 0.0:
        pred_pos = scores >= th
        tp = int((pred_pos & labels_bool).sum())
        npp = int(pred_pos.sum())
        if npp == 0:
            ps.append(1.0); rs.append(0.0); ths.append(th)
        else:
            ps.append(tp / npp); rs.append(tp / num_cond_pos); ths.append(th)
        th -= 0.0005
    return np.array(ps), np.array(rs), np.array(ths)


ps_v2, rs_v2, ths_v2 = pr_sweep(v2_mean, lbl_e)
ps_yu, rs_yu, ths_yu = pr_sweep(yu_mean, lbl_e)

auc_v2 = metrics.auc(rs_v2, ps_v2)
auc_yu = metrics.auc(rs_yu, ps_yu)
print(f'v2 PR-AUC: {auc_v2:.4f}   max R: {rs_v2.max():.3f}   max P: {ps_v2.max():.3f}')
print(f'Yu PR-AUC: {auc_yu:.4f}   max R: {rs_yu.max():.3f}   max P: {ps_yu.max():.3f}')

fig, ax = plt.subplots(figsize=(6, 3.7), dpi=200)
for side in ('top', 'right', 'left', 'bottom'):
    ax.spines[side].set_color('#808080')
ax.tick_params(direction='in', color='#808080')
plt.grid(color='#c0c0c0', linestyle='--', linewidth=0.5)
plt.xlabel('Recall', fontweight='bold'); plt.ylabel('Precision', fontweight='bold')
plt.xlim(0, 1); plt.ylim(0, 1)
plt.plot(rs_v2, ps_v2, label=f'v2 (AUC={auc_v2:.3f})')
plt.plot(rs_yu, ps_yu, label=f'Yu (AUC={auc_yu:.3f})')
plt.legend(loc='lower left')
plt.title('PR curves — disp_E binary')
plt.show()

## 5. Summary metrics at matched recall
100%-recall threshold for each model, plus precision there.

In [ ]:
def summary(name, ps, rs, ths):
    i = len(rs) - 1
    while i > 0 and rs[i] == 1.0:
        i -= 1
    i += 1
    if i >= len(rs):
        i = len(rs) - 1
    return {
        'model': name,
        'PR_AUC': metrics.auc(rs, ps),
        'P@100R': ps[i],
        'threshold@100R': ths[i],
    }


pd.DataFrame([summary('v2', ps_v2, rs_v2, ths_v2),
              summary('Yu', ps_yu, rs_yu, ths_yu)])

## 6. Binary confusion matrices (vote-count methodology)
Per-model threshold + ensemble vote, as in `Predict-fa1t-38-test.ipynb`:

- **v2** — for each of 10 models, vote `disp_e` iff `disp_e >= 0.215`. `disp_e_p` = count of models voting `disp_e`. Predict positive when `disp_e_p > 0`.
- **Yu** — for each of 10 models, vote positive iff sigmoid `>= 0.5`. `disp_e_p` = count. Predict positive when `disp_e_p > 0`.

This matches the original v2 notebook exactly (so the v2 confusion matrix below should reproduce the numbers there).

In [ ]:
from sklearn.metrics import confusion_matrix

VOTE_TH_V2 = 0.215   # matches Predict-fa1t-38-test.ipynb
VOTE_TH_YU = 0.5     # standard sigmoid cutoff

v2_disp_e_p = np.zeros(len(ids), dtype=int)
for preds in v2_ensemble_preds:
    for ex_id, row in preds.iterrows():
        if ex_id in index:
            if row[labels[col_e]] >= VOTE_TH_V2:
                v2_disp_e_p[index[ex_id]] += 1

yu_disp_e_p = (yu_mat >= VOTE_TH_YU).sum(axis=0)

v2_pos = v2_disp_e_p > 0
yu_pos = yu_disp_e_p > 0

bin_labels = ['not disp_E', 'disp_E']
y_true_bin = lbl_e.astype(int)


def show_cm(name, y_pred_bin, note):
    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    cm_df = pd.DataFrame(cm,
                         index=[f'true {l}' for l in bin_labels],
                         columns=[f'pred {l}' for l in bin_labels])
    print(f'=== {name}  {note} ===')
    print(cm_df)
    print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    print(f'Precision: {prec:.4f}   Recall: {rec:.4f}')

    fig, ax = plt.subplots(figsize=(4.5, 4), dpi=150)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(bin_labels); ax.set_yticklabels(bin_labels)
    ax.set_xlabel('Predicted', fontweight='bold')
    ax.set_ylabel('True', fontweight='bold')
    ax.set_title(f'{name} — disp_E  ({note})')

    thresh_color = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh_color else 'black')

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


show_cm('v2', v2_pos.astype(int), f'per-model th={VOTE_TH_V2}, vote>0')
show_cm('Yu', yu_pos.astype(int), f'per-model th={VOTE_TH_YU}, vote>0')

## 7. Disagreement set
TCEs where v2 and Yu disagree under the vote-count methodology above (`disp_e_p > 0`).

In [ ]:
disagree = v2_pos != yu_pos

tic_by_id = tce_labels['TIC ID']
rows = []
for ex_id, idx in index.items():
    if disagree[idx]:
        rows.append({
            'astro_id': ex_id,
            'TIC ID': tic_by_id.get(ex_id, None),
            'truth_disp_E>0': bool(lbl_e[idx]),
            'v2_votes': int(v2_disp_e_p[idx]),
            'yu_votes': int(yu_disp_e_p[idx]),
            'v2_pred': bool(v2_pos[idx]),
            'yu_pred': bool(yu_pos[idx]),
        })
disagree_df = pd.DataFrame(rows).sort_values(
    ['truth_disp_E>0', 'v2_votes'], ascending=[False, False])
print(f'{len(disagree_df)} disagreements out of {len(ids)}')
disagree_df.head(20)